In [1]:
import pandas as pd
import os

In [2]:
chain2file = {}
for dirname,dirs,filenames in os.walk("./pep_data"):
    for filename in filenames:
        chain = dirname.split("/")[-1]
        if chain not in chain2file.keys():
            chain2file[chain] = [os.path.join(dirname,filename)]
        else:
             chain2file[chain].append(os.path.join(dirname,filename))

In [7]:
import parmap


def read_csv(path):
    df = pd.read_csv(path,usecols=["CDR3(pep)","copy"])
    if df.shape[0]==0:
        return pd.DataFrame(columns=["CDR3(pep)","copy","CDR3_Lenth"])
    df = df.groupby("CDR3(pep)").sum().reset_index().sort_values(by="copy",ascending=False)
    df["CDR3_Lenth"] = df["CDR3(pep)"].str.len()
    CDR3_lenth_file = "CDR3_lenth_distribution/"+path.split("__")[-1].split(".csv")[0]
    df.to_csv(CDR3_lenth_file+"/"+path.split("/")[-1],index=False)
    df_lenth_distribution = df[["copy","CDR3_Lenth"]].groupby(by="CDR3_Lenth").sum()
    df_lenth_distribution = df_lenth_distribution/df_lenth_distribution.sum()
    df_lenth_distribution.columns= [path.split("/")[-1]]
    df_lenth_distribution = df_lenth_distribution.reset_index()
    return df_lenth_distribution
    

chain2df = {}
for chain,paths in chain2file.items():
    CDR3_lenth_file = "CDR3_lenth_distribution/"+chain
    if not os.path.exists(CDR3_lenth_file):
        os.makedirs(CDR3_lenth_file)
    result = parmap.map_async(read_csv,paths,pm_processes=256)
    result.wait()
    output = result.get()
    chain2df[chain] = output


category = "HHY"
file_name = "./Profile_All_24_04_22.csv"

df_all = pd.DataFrame(columns=["Sample"])
for chain,dfs_lenth_distribution in chain2df.items():
    df = pd.DataFrame(columns=["CDR3_Lenth"])
    for df_lenth_distribution in dfs_lenth_distribution:
        df = pd.merge(df,df_lenth_distribution,on="CDR3_Lenth",how="outer")
        df = df.sort_values(by="CDR3_Lenth",ascending=True)
        df = df.fillna(0)
    df_len_matrix = df.copy()
    df_len_matrix.index = df_len_matrix.pop("CDR3_Lenth")
    df_len_matrix = df_len_matrix.T.reset_index()
    df_len_matrix["index"] = df_len_matrix["index"].apply(lambda x:x.split("__")[0])
    df_len_matrix.insert(loc=0,column="Sample",value=df_len_matrix.pop("index"))
    df_dp = pd.read_csv(file_name,usecols=["Sample",category])
    df_dp = df_dp.dropna()
    df_len_matrix = pd.merge(df_dp,df_len_matrix,how="inner",on="Sample")
    df_len_matrix.to_csv("CDR3_distribution_matrix_"+chain+".csv",index=False)

    mean_lenth_df = pd.DataFrame(df[df.columns[1:]].apply(lambda x: x*df["CDR3_Lenth"],axis=0).sum()).reset_index()
    mean_lenth_df.columns = ["Sample",chain+"_meanCDR3_len"]
    mean_lenth_df["Sample"] = mean_lenth_df["Sample"].apply(lambda x:x.split("__")[0])
    df_all = pd.merge(df_all,mean_lenth_df,how="outer",on="Sample")
df_dp = pd.read_csv(file_name,usecols=["Sample",category])
df_dp = df_dp.dropna()
df_all = pd.merge(df_dp,df_all,how="inner",on="Sample")
df_all.to_csv("CDR3_lenth.csv",index=False)